# ByteEmbed — Low-Resource Byte-vs-Subword Study (A100)

Clean iso-compute study: **byt5 vs mt5** × {small, base, large} distilled from a frozen **SONAR**
teacher (1024-d, 200-lang → no teacher-ceiling) on **9 languages** — 6 low-resource
(Telugu, Tamil, Marathi, Amharic, Hausa, Kinyarwanda) + 3 high-resource anchors (English, Mandarin,
Arabic).

**Uniform eval battery** (every language): SIB-200 (classification) · Belebele (retrieval) ·
FLORES-1012 (parallel bitext) · STS · + MIRACL deep retrieval (en/zh/ar/te) · + a tokenization-
efficiency / compute-cost table.

**Everything is resumable**: results save after each model, `*-large` checkpoints model+optimizer,
teacher targets are cached (one pass), and re-running a cell skips finished work. Run top-to-bottom;
do the smoke cell first.

### 1. GPU check — confirm you're on an A100 (Runtime → Change runtime type → A100)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')

### 2. Clone repo + install deps
Imports work from the repo root even if the editable install is skipped.

In [ ]:
import os
os.chdir('/content')
REPO = 'https://github.com/Aarushvinod/embedding-research.git'
if not os.path.isdir('/content/embedding-research'):
    !git clone -q $REPO
os.chdir('/content/embedding-research')
!git pull -q
!pip install -q -r requirements-cloud.txt
!pip install -q -e . || echo '(editable install skipped — running from repo root is fine)'
print('setup done | cwd', os.getcwd())

### 3. SONAR teacher install (fault-tolerant)
`fairseq2` wheels are torch/CUDA-specific, so this is a SEPARATE cell. **If it fails, that's OK** —
`byte_embed.teachers.load_teacher()` falls back to **LaBSE** automatically (the students train against
cached targets, so the pipeline is teacher-agnostic). Check the printed teacher in the smoke cell.

In [ ]:
# Try SONAR; do not let a failure abort the run (we fall back to LaBSE).
!pip install -q sonar-space fairseq2 || echo 'SONAR install failed — will fall back to LaBSE'
try:
    from sonar.inference_pipelines.text import TextToEmbeddingModelPipeline
    print('SONAR import OK — teacher will be SONAR (1024-d)')
except Exception as e:
    print('SONAR unavailable (%s) — teacher will fall back to LaBSE (768-d)' % type(e).__name__)

### 4. Persist results + checkpoints to Drive
So a Colab disconnect doesn't lose the cached teacher targets / checkpoints / results. **Skip this
cell** to run on ephemeral disk. Set `HF_TOKEN` if you hit Hub rate limits.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, shutil
PERSIST = '/content/drive/MyDrive/byteembed_lowres'
for d in ('results', 'checkpoints'):
    os.makedirs(f'{PERSIST}/{d}', exist_ok=True)
    if not os.path.islink(d):
        if os.path.isdir(d): shutil.rmtree(d)
        os.symlink(f'{PERSIST}/{d}', d)
print('persisting results/ and checkpoints/ to', PERSIST)
# from huggingface_hub import login; login()   # uncomment + run if you hit HF rate limits

### 5. Smoke test (~5 min) — validate the WHOLE pipeline first
3 langs (am/rw/en), 2 tiny students, tiny eval. If this prints a table, everything works
(teacher load → balanced data → cached targets → train → SIB/Belebele/FLORES/STS/MIRACL → efficiency → save).
**Confirm which teacher loaded (SONAR vs LaBSE) in the log.**

In [ ]:
from byte_embed.run_lowresource import run
_ = run(smoke=True, out='results/byte_lowresource_smoke.json')

### 6. Tokenization-efficiency table (fast, no training)
The motivation metric: subword token tax vs byte UTF-8 tax (vs English, on parallel FLORES-1012).
Honest — byte costs MORE for non-Latin scripts (UTF-8 multibyte); the byte case rests on quality-per-
parameter, not on being cheaper.

In [ ]:
from byte_embed.efficiency import fertility_table, print_fertility
from byte_embed.config import STUDY_LANGS
print_fertility(fertility_table(STUDY_LANGS))

### 7. Full study — 6 students (byt5/mt5 × small/base/large), resumable
~16–24 h across the size range (large ≈ 5–7 h each). Re-run to resume (finished models skip; teacher
targets + `*-large` checkpoints are cached). `n_per_lang=42000` (Kinyarwanda floor).

In [ ]:
from byte_embed.run_lowresource import run
_ = run(out='results/byte_lowresource.json')

### 8. Results — table + figures

In [ ]:
import json
from byte_embed.run_lowresource import _summary
res = json.load(open('results/byte_lowresource.json'))
_summary(res)

# figures: tokenization tax, quality-per-param Pareto, byte-minus-subword vs fertility
import gen_figures
gen_figures.main_lowres()
print('figures written to figures/')

### 9. Download results + figures
Already on Drive if you ran cell 4. Otherwise grab them here.

In [ ]:
from google.colab import files
files.download('results/byte_lowresource.json')
for f in ('fig_lowres_tax.png', 'fig_lowres_quality_params.png', 'fig_lowres_delta_fertility.png'):
    try: files.download(f'figures/{f}')
    except Exception as e: print('skip', f, e)